7_Statlog_(Australian_Credit_Approval)

¿Quién lo creó y cómo se obtuvieron los datos?
El dataset fue enviado por Ross Quinlan al UCI Machine Learning Repository en 1987. Al igual que el Credit Approval anterior, proviene de registros reales de solicitudes de tarjetas de crédito de una institución financiera australiana. Los datos fueron también completamente anonimizados para proteger la privacidad de los solicitantes. Este dataset existe en el repositorio relacionado con el anterior (Credit Screening Database), pero en una versión ligeramente diferente con etiquetas numéricas en lugar de letras.

¿De qué trata?
Esta base de datos existe en el repositorio en una forma ligeramente diferente al Credit Screening Dataset. Contiene solicitudes de crédito de una institución australiana real, con las mismas categorías de información pero con las etiquetas categóricas recodificadas en valores numéricos para facilitar el procesamiento por algoritmos estadísticos.

¿Qué contiene?
Hay 6 atributos numéricos y 8 atributos categóricos. Las etiquetas fueron cambiadas para conveniencia de los algoritmos estadísticos. Por ejemplo, el atributo 4 originalmente tenía 3 etiquetas (p, g, gg) y fueron cambiadas a (1, 2, 3). A1 es binaria (0,1), A2 continua, A3 continua, A4 categórica con 3 niveles, A5 con 14 niveles, A6 con 9 niveles. Contiene 690 observaciones. La variable objetivo es también binaria (aprobado/rechazado), codificada como 0 o 1.

Objetivo del modelo
Clasificación binaria: predecir la aprobación o rechazo de una solicitud de crédito. A diferencia del dataset anterior (Credit Approval UCI), este ya tiene las variables categóricas convertidas a números, lo que facilita algunos pasos del preprocesamiento. Es útil para comparar directamente el rendimiento de distintos algoritmos en datos de crédito con diferente codificación. Una práctica habitual es comparar ambos datasets (Credit Approval y Australian Credit) para analizar cómo la codificación de variables impacta el rendimiento de los modelos.

In [1]:
# ============================================================
# LIBRERÍAS GENERALES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [2]:
# ── PASO 1: CARGA ───────────────────────────────────────────
nombres_au = ['A1','A2','A3','A4','A5','A6','A7',
              'A8','A9','A10','A11','A12','A13','A14','target']

df_aus = pd.read_csv('Datasets/7_Statlog_(Australian_Credit_Approval)/australian.dat',
                      sep=' ',
                      header=None,
                      names=nombres_au)

print('Shape:', df_aus.shape)
print('Nulos:', df_aus.isnull().sum().sum(), '← no hay ninguno')
print('\nTarget:')
print(df_aus['target'].value_counts())
print('(0=rechazado, 1=aprobado)')

Shape: (690, 15)
Nulos: 0 ← no hay ninguno

Target:
target
0    383
1    307
Name: count, dtype: int64
(0=rechazado, 1=aprobado)


In [3]:
# ── PASO 2: CONSTRUIR X e y ─────────────────────────────────
# Ya todo es numérico, no hay que convertir nada
X_raw_aus = df_aus.drop(columns='target').values.astype(float)
y_aus     = df_aus['target'].values.astype(float)
m_aus     = y_aus.size

print('X shape:', X_raw_aus.shape)
print('y shape:', y_aus.shape)

X shape: (690, 14)
y shape: (690,)


In [4]:
# ============================================================
# FUNCIÓN DE BALANCEO — oversampling con numpy
# ============================================================
def balancear(X, y):
    """
    Balancea un dataset desbalanceado usando OVERSAMPLING.
    
    ¿Qué hace?
    - Identifica cuántos ejemplos tiene cada clase
    - La clase con MÁS ejemplos queda igual
    - Las clases con MENOS ejemplos se repiten (con reemplazo)
      hasta tener la misma cantidad que la clase mayoritaria
    - Al final todas las clases tienen el mismo número de filas
    
    ¿Por qué oversampling y no undersampling?
    - Undersampling borra filas → perdemos información
    - Oversampling agrega filas → mantenemos toda la información original
    """
    clases = np.unique(y)
    n_max  = max(np.sum(y == c) for c in clases)   # tamaño de la clase más grande
    
    X_bal_list = []
    y_bal_list = []
    
    for c in clases:
        idx    = np.where(y == c)[0]               # índices de esta clase
        n_c    = len(idx)                           # cuántos ejemplos tiene
        
        if n_c < n_max:
            # repetir filas hasta alcanzar n_max
            extra  = n_max - n_c
            idx_extra = np.random.choice(idx, size=extra, replace=True)
            idx_final = np.concatenate([idx, idx_extra])
        else:
            idx_final = idx
        
        X_bal_list.append(X[idx_final])
        y_bal_list.append(y[idx_final])
    
    X_bal = np.concatenate(X_bal_list, axis=0)
    y_bal = np.concatenate(y_bal_list, axis=0)
    
    # Mezclar aleatoriamente para no dejar todas las clases juntas
    perm  = np.random.permutation(len(y_bal))
    return X_bal[perm], y_bal[perm]

def mostrar_balance(y, nombre, antes_despues='ANTES'):
    """Imprime cuántos ejemplos tiene cada clase."""
    clases, cuentas = np.unique(y, return_counts=True)
    print(f'  Balance {antes_despues} — {nombre}:')
    for c, n in zip(clases, cuentas):
        print(f'    Clase {int(c)}: {n} ({n/len(y)*100:.1f}%)')

np.random.seed(42)   # para reproducibilidad
print('Funciones de balanceo definidas')

Funciones de balanceo definidas


In [5]:
mostrar_balance(y_aus, 'Australian Credit', 'ANTES')

  Balance ANTES — Australian Credit:
    Clase 0: 383 (55.5%)
    Clase 1: 307 (44.5%)


In [6]:
def featureNormalize(X):
    """
    Normaliza las features de X.
    Para cada columna: resta la media y divide por la desviación estándar.
    
    Retorna:
      X_norm : X normalizado (mismo tamaño que X)
      mu     : media de cada columna (se guarda para normalizar datos nuevos)
      sigma  : desviación estándar de cada columna
    """
    X_norm = X.copy()
    mu     = np.mean(X, axis=0)   # media de cada columna
    sigma  = np.std(X, axis=0)    # desviación estándar de cada columna
    X_norm = (X - mu) / sigma     # estandarización Z-score
    return X_norm, mu, sigma

In [8]:
# ── PASO 3: NORMALIZAR y COLUMNA DE UNOS ────────────────────
X_norm_aus, mu_aus, sigma_aus = featureNormalize(X_raw_aus)
X_bal_au, y_bal_au = balancear(X_norm_aus, y_aus)
mostrar_balance(y_bal_au, 'Australian Credit', 'DESPUÉS')

X_aus = np.concatenate([np.ones((len(y_bal_au), 1)), X_bal_au], axis=1)
y_aus  = y_bal_au

print('X_aus final:', X_aus.shape)
print('X:', X_aus.shape, '| y:', y_aus.shape)

  Balance DESPUÉS — Australian Credit:
    Clase 0: 383 (50.0%)
    Clase 1: 383 (50.0%)
X_aus final: (766, 15)
X: (766, 15) | y: (766,)
